# Laser Position PV Archiver Probe

This notebook tests each `laser / Position` PV marked `measurement: true` in `pv_groups.yaml`. For each configured PV, it probes both the base name and the `1H` sampled name against the archiver API.

In [1]:
from __future__ import annotations

import datetime as dt
import sys
import time
from pathlib import Path
from zoneinfo import ZoneInfo

import requests
import yaml

try:
    import pandas as pd
except ImportError:
    pd = None

APP_DIR = Path.cwd()
if not (APP_DIR / "pv_groups.yaml").exists():
    APP_DIR = Path("sparklines/app").resolve()

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

try:
    from sparklines_hierarchy import ARCHIVER_URL, LOCAL_TIMEZONE, _format_archive_time
except ImportError:
    ARCHIVER_URL = "http://lcls-archapp.slac.stanford.edu/retrieval/data/getData.json"
    LOCAL_TIMEZONE = ZoneInfo("America/Los_Angeles")

    def _format_archive_time(value, *, local_timezone=LOCAL_TIMEZONE):
        if value.tzinfo is None:
            value = value.replace(tzinfo=local_timezone)
        return value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

PV_GROUPS_PATH = APP_DIR / "pv_groups.yaml"
PV_GROUPS_PATH

PosixPath('/sdf/home/a/aaron73/xleap/sparklines/app/pv_groups.yaml')

In [2]:
def iter_laser_position_measurements(groups_path: Path = PV_GROUPS_PATH):
    payload = yaml.safe_load(groups_path.read_text(encoding="utf-8")) or {}
    for group in payload.get("groups", []):
        if group.get("group_name") != "laser":
            continue
        position = group.get("subgroups", {}).get("Position", {})
        for spec in position.get("pv", []):
            if spec.get("measurement") is True:
                yield spec["pv_name"]


def base_and_1h_candidates(pv_name: str) -> list[dict[str, str]]:
    base = pv_name[:-2] if pv_name.endswith("1H") else pv_name
    sampled = f"{base}1H"
    return [
        {"configured_pv": pv_name, "candidate_kind": "base", "pv_name": base},
        {"configured_pv": pv_name, "candidate_kind": "1H", "pv_name": sampled},
    ]


configured_pvs = list(iter_laser_position_measurements())
probe_specs = [candidate for pv in configured_pvs for candidate in base_and_1h_candidates(pv)]

configured_pvs, probe_specs

(['BPMS:GUNB:314:FW:X_SLOW',
  'BPMS:GUNB:314:FW:Y_SLOW',
  'BPMS:IN20:221:XCU1H',
  'BPMS:IN20:221:YCU1H'],
 [{'configured_pv': 'BPMS:GUNB:314:FW:X_SLOW',
   'candidate_kind': 'base',
   'pv_name': 'BPMS:GUNB:314:FW:X_SLOW'},
  {'configured_pv': 'BPMS:GUNB:314:FW:X_SLOW',
   'candidate_kind': '1H',
   'pv_name': 'BPMS:GUNB:314:FW:X_SLOW1H'},
  {'configured_pv': 'BPMS:GUNB:314:FW:Y_SLOW',
   'candidate_kind': 'base',
   'pv_name': 'BPMS:GUNB:314:FW:Y_SLOW'},
  {'configured_pv': 'BPMS:GUNB:314:FW:Y_SLOW',
   'candidate_kind': '1H',
   'pv_name': 'BPMS:GUNB:314:FW:Y_SLOW1H'},
  {'configured_pv': 'BPMS:IN20:221:XCU1H',
   'candidate_kind': 'base',
   'pv_name': 'BPMS:IN20:221:XCU'},
  {'configured_pv': 'BPMS:IN20:221:XCU1H',
   'candidate_kind': '1H',
   'pv_name': 'BPMS:IN20:221:XCU1H'},
  {'configured_pv': 'BPMS:IN20:221:YCU1H',
   'candidate_kind': 'base',
   'pv_name': 'BPMS:IN20:221:YCU'},
  {'configured_pv': 'BPMS:IN20:221:YCU1H',
   'candidate_kind': '1H',
   'pv_name': 'BPMS:IN20:

In [3]:
end = dt.datetime.now(tz=LOCAL_TIMEZONE)
start = end - dt.timedelta(hours=1)
timeout = 20.0

print("Archiver URL:", ARCHIVER_URL)
print("Window:", _format_archive_time(start, local_timezone=LOCAL_TIMEZONE), "to", _format_archive_time(end, local_timezone=LOCAL_TIMEZONE))

Archiver URL: http://lcls-archapp.slac.stanford.edu/retrieval/data/getData.json
Window: 2026-05-26T18:01:40.000Z to 2026-05-26T19:01:40.000Z


In [4]:
def unwrap_archive_payload(raw_payload):
    if isinstance(raw_payload, list):
        if not raw_payload:
            return None
        return raw_payload[0]
    return raw_payload


def probe_archive_pv(pv_name: str, start, end, *, timeout: float = timeout):
    params = {
        "pv": pv_name,
        "from": _format_archive_time(start, local_timezone=LOCAL_TIMEZONE),
        "to": _format_archive_time(end, local_timezone=LOCAL_TIMEZONE),
    }
    request_start = time.monotonic()
    response = requests.get(ARCHIVER_URL, params=params, timeout=timeout)
    elapsed_s = time.monotonic() - request_start
    response.raise_for_status()

    payload = unwrap_archive_payload(response.json())
    if not payload:
        return {
            "ok": False,
            "point_count": 0,
            "elapsed_s": elapsed_s,
            "error": "empty payload",
        }

    data = payload.get("data")
    if data is None:
        return {
            "ok": False,
            "point_count": 0,
            "elapsed_s": elapsed_s,
            "error": "payload missing data array",
        }

    seconds = [item.get("secs") for item in data if isinstance(item, dict) and item.get("secs") is not None]
    return {
        "ok": bool(data),
        "point_count": len(data),
        "elapsed_s": elapsed_s,
        "first_time": dt.datetime.fromtimestamp(min(seconds), tz=LOCAL_TIMEZONE) if seconds else None,
        "last_time": dt.datetime.fromtimestamp(max(seconds), tz=LOCAL_TIMEZONE) if seconds else None,
        "error": "" if data else "no data points in window",
    }


rows = []
for spec in probe_specs:
    row = dict(spec)
    try:
        row.update(probe_archive_pv(spec["pv_name"], start, end))
    except Exception as exc:
        row.update({"ok": False, "point_count": 0, "elapsed_s": None, "error": repr(exc)})
    rows.append(row)

if pd is not None:
    results = pd.DataFrame(rows)
    display(results)
else:
    results = rows
    for row in rows:
        print(row)

,configured_pv,candidate_kind,pv_name,ok,point_count,elapsed_s,first_time,last_time,error
0,BPMS:GUNB:314:FW:X_SLOW,base,BPMS:GUNB:314:FW:X_SLOW,True,1,1.231830,2025-12-14 23:02:58-08:00,2025-12-14 23:02:58-08:00,
1,BPMS:GUNB:314:FW:X_SLOW,1H,BPMS:GUNB:314:FW:X_SLOW1H,False,0,NaN,NaT,NaT,HTTPError('404 Client Error: Not Found for url...
2,BPMS:GUNB:314:FW:Y_SLOW,base,BPMS:GUNB:314:FW:Y_SLOW,True,1,0.015077,2025-12-14 23:02:58-08:00,2025-12-14 23:02:58-08:00,
3,BPMS:GUNB:314:FW:Y_SLOW,1H,BPMS:GUNB:314:FW:Y_SLOW1H,False,0,NaN,NaT,NaT,HTTPError('404 Client Error: Not Found for url...
4,BPMS:IN20:221:XCU1H,base,BPMS:IN20:221:XCU,False,0,NaN,NaT,NaT,HTTPError('404 Client Error: Not Found for url...
5,BPMS:IN20:221:XCU1H,1H,BPMS:IN20:221:XCU1H,True,2543,0.013684,2026-05-26 11:01:39-07:00,2026-05-26 11:55:12-07:00,
6,BPMS:IN20:221:YCU1H,base,BPMS:IN20:221:YCU,False,0,NaN,NaT,NaT,HTTPError('404 Client Error: Not Found for url...
7,BPMS:IN20:221:YCU1H,1H,BPMS:IN20:221:YCU1H,True,2544,0.022637,2026-05-26 11:01:39-07:00,2026-05-26 11:55:12-07:00,


In [5]:
if pd is not None:
    display(
        results.pivot_table(
            index="configured_pv",
            columns="candidate_kind",
            values="point_count",
            aggfunc="first",
            fill_value=0,
        )
    )

candidate_kind,1H,base
configured_pv,,
BPMS:GUNB:314:FW:X_SLOW,0,1
BPMS:GUNB:314:FW:Y_SLOW,0,1
BPMS:IN20:221:XCU1H,2543,0
BPMS:IN20:221:YCU1H,2544,0
